In [8]:
import numpy as np
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib import pyplot as plt
import toml
from pathlib import Path
import os
import glob

## Get Images

In [ ]:
def contains_target(frames, filenames, coords:SkyCoord, fk4_coords=None, n_cutoff=3, arcsec_threshold=100):

    n_target_frames = 0

    for i, frame in enumerate(frames):

        conditions = [
            frame["CAMNAME"] == "narrow",
            frame["GRSNAME"] == "clear",
            not (
                "SLITNAME" in frame and (
                    "vortex" in frame["SLITNAME"] or
                    "corona" in frame["SLITNAME"]
                )
            )
        ]

        if not np.all(conditions):
            print(f'[Warning]: Skipping file {filenames[i]} due to conditions')
            continue

        if (type(frame["RA"]) != str) or (type(frame["DEC"]) != str):
            print('[Warning]: Skipping file $(filenames[i]) due to missing RA/DEC')
            continue

        if frame["RADECSYS"] == "FK4":
            print(f'[Info]: Using FK4 coordinates for {filenames[i]}')
            if fk4_coords != None:
                ra, dec = fk4_coords
            else:
                print("[Warning]: Files are in FK4 but no Fk4 coordinates provided!")
        
        radec_distance = coords.separation(SkyCoord(frame["RA"], frame["DEC"], unit=(u.hourangle, u.deg))).degree * 3600
        
        print(f'[Info]: Coordinates: object={frame["OBJECT"]} target={frame["TARGNAME"]} ra={ra} dec={dec} frame_ra={frame["RA"]} frame_dec={frame["DEC"]} radec_distance={radec_distance} radecsys={frame["RADECSYS"]}')

        if radec_distance < arcsec_threshold and frame["SHRNAME"].lower() == "open":
            print(f'RADEC FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]} radec_distance={radec_distance}')
            n_target_frames += 1

    if n_target_frames <= n_cutoff:
        print(f'[Error]: Not enough science frames found (only found {n_target_frames}), skipping obslog generation')
        return False
    else: 
        return True

## Sort Frames

In [49]:
def sort_frames(frames, filenames, lampoff_threshold=100.0, arcsec_threshold=100.0, filter_names=None):

    flat_el = 45.0 # degrees, elevation of the flat field frames

    sci = []
    flats = []
    flats_sky = []
    flats_lampon = []
    flats_lampoff = []
    darks = []

    for i, frame in enumerate(frames):

        if filter_names != None:
            if (filter_names in frame["OBJECT"]):
                print('[Info]: SKIPPED DUE TO FILTER NAME')
                continue

        p1 = SkyCoord(0, flat_el, unit=u.deg)
        p2 = SkyCoord(0, frame["EL"], unit=u.deg)
        altaz_distance = p2.separation(p1).degree * 3600

        if (altaz_distance <  arcsec_threshold) and (frame["WCDMSTAT"].lower() == "open" or frame["WCDMSTAT"].lower() == "idle") and (frame["WCDTSTAT"].lower() == "open" or frame["WCDTSTAT"].lower() == "idle"):

            if np.median(frame) < lampoff_threshold:
                print(f'[Info]: LAMPOFF FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]} altaz_distance={altaz_distance}')
                flats_lampoff.append(filenames[i])
            else:
                print(f'[Info]: LAMPON FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]} altaz_distance={altaz_distance}')
                flats_lampon.append(filenames[i])

        elif "sky" in frame["OBJECT"].lower() or "twi" in frame["OBJECT"].lower():

            print(f'[Info]: SKY FLAT FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]}')
            flats_sky.append(filenames[i])

        elif frame["SHRNAME"] == "closed":

            print(f'[Info]: DARK FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]}')
            darks.append(filenames[i])
        
        else:

            print(f'SCI FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]}')
            sci.append(filenames[i])

    return sci, flats, flats_sky, flats_lampon, flats_lampoff, darks

## Make Observing Log

In [ ]:
def make_obslog(data_folder:str, date, obslog_filepath, sci, flats, flats_sky, flats_lampon, flats_lampoff, darks):

    obslog = {
        "data_folder":data_folder,
        "date":date,
        "raw":{
            "sci":sci,
            "flats":flats,
            "flats_sky":flats_sky,
            "flats_lampon":flats_lampon,
            "flats_lampoff":flats_lampoff,
            "darks":darks
        }
    }

    toml_string = toml.dumps(obslog)
    output_file = Path('obslog_filepath')
    
    with open(output_file, "w") as toml_file:
        toml.dump(obslog, toml_file)

In [9]:
def generate_obslogs_generic():

    date = "2025-10-08"
    observation_folder = "/Users/jsn/landing/projects/AIR.jl/data/polmode/"

    output_folder = "/Users/jsn/landing/projects/AIR.jl/polmode/obslogs"
    Path(output_folder).mkdir(exist_ok=True)

    data_folder = Path(observation_folder) / Path(date)

    frames = []
    filenames = []

    for filename in glob.glob(Path(data_folder) / Path('raw') / Path('*.fits'), recursive=True):
        frames.append(fits.getdata('filename'))
        filenames.append(Path(data_folder) / Path('raw') / Path(filename))

    sorted_frames = sort_frames(frames, filenames, filter_names="pol_cal")
    make_obslog(data_folder, date, Path(output_folder) / Path(f'${date}_obslog.toml'), sorted_frames)

In [ ]:
generate_obslogs_generic()

## Bad Pixel Replacer

In [ ]:
# bad pixel replacer
def median(data:np.array, mask:float, median_size, fail_val:float=0.0):
    '''
    Replaces bad pixels with the median of a box around them.
    '''
    bad_indicies = np.array(np.where(data < mask)).T
    half_size = np.floor(median_size / 2)

    h, w = data.shape
    h_i = h - 1
    w_i = w - 1
    
    for index in bad_indicies:

        i, j = index
        min_i = int(max(0, i-half_size))
        max_i = int(min(i+half_size, w_i) + 1)
        min_j = int(max(0, j-half_size))
        max_j = int(min(j+half_size, h_i) + 1)

        # if no good pixels, use fail_val
        if np.all(data[min_i:max_i, min_j:max_j] < mask):
            data[i,j] = fail_val
        else:
            data[i,j] = np.median(data[min_i:max_i, min_j:max_j])

## Make Darks

In [ ]:
def make_masters(frames, keylist, n_sigma:float=6.0, median_size:int=7, method:callable=median, min_frames:int=3):

    frame_dict = dict(zip(keylist, frames))

    for key in list(frame_dict.keys()):
        if len(frame_dict[key]) < min_frames:
            print(f'[Warning]: Not enough frames for key {key}, skipping...')
            del(frame_dict[key])
        else:
            print(f'[Info]: Making master for key -> {key}, count -> {(len(frame_dict[key]))}')

    master_frames = {}
    master_frames_masks = {}

    # for key in frame_dict.keys():

    #     if len(frame_dict[key]) < min_frames:
    #         print(f'[Warning]: Not enough frames for key {key}, skipping...')
    #         continue

    #     # applies method (mean, median) to frame list
    #     stack = framelist_to_cube(frame_dict[key])
    #     method_stack = method(stack, dims=3) |> x -> dropdims(x, dims=3)
    #     mf = AstroImage(method_stack)

    #     # take each frame, make a sigma clip mask, and then combine them or-wise
    #     masks = [make_sigma_clip_mask(frame.data, n_sigma) for frame in frame_dict[key]]
    #     sigma_clip_mask = reduce(.|, masks)

    #     # crop the original bad pixel mask to the size of the median frame
    #     bad_pixel_mask = copy(NIRC2_bad_pixel_mask)
    #     if size(bad_pixel_mask) != size(mf)
    #         bad_pixel_mask, _, _ = crop(NIRC2_bad_pixel_mask, size(mf))
    #     end

    #     # combine masks
    #     mask = bad_pixel_mask .| sigma_clip_mask

    #     # make the median frame to do pixel replacement
    #     # not super efficient, but it works
    #     median_mf = mapwindow(median, mf.data, (median_size, median_size))

    #     # finally, assign the median values to the masked pixels
    #     mf.data[mask] .= median_mf[mask]

    #     # repopulate the header with the key values so we can find the keys later
    #     for (i, k) in enumerate(keylist)
    #         mf[k] = key[i]
    #     end

    #     mf["FILENAME"] = frames[1]["FILENAME"] # copy the first file name to the master
    #     mf["NFRAMES"] = length(frame_dict[key])
    #     mf["NSIGMASK"] = n_sigma
    #     mf["MEDSIZE"] = median_size
    #     mf["NPIXMASK"] = sum(mask)
    #     mf["MAMEDIAN"] = median(mf.data[.!mask])
    #     mf["MAMEAN"] = mean(mf.data[.!mask])
    #     mf["MASTD"] = std(mf.data[.!mask])

    #     master_frames[key] = mf
    #     master_frames_masks[key] = mask

    return master_frames, master_frames_masks

In [ ]:
def make_darks(darks_frames, darks_keylist=["NAXIS1", "NAXIS2", "ITIME", "COADDS"]):

    if not darks_frames:
        master_darks, master_darks_masks = make_masters(darks_frames, darks_keylist, method=median)
    else:
        master_darks = {}
        master_darks_masks = {}

    return master_darks, master_darks_masks,